# 03 — Customer Segmentation

## Purpose

This notebook segments customers using the customer-level feature table produced by:

```text
src/feature_engineering.py
        ↓
data/processed/customer_features.csv
```

The workflow compares an interpretable **RFM-based segmentation** with an unsupervised **K-Means clustering** approach.

The goal is to inspect the customer feature space, establish an RFM baseline, select a reasonable number of clusters, profile K-Means clusters, and save a reproducible segmentation artifact.

## Business questions

- Who are the highest-value customers?
- Which customers are highly active?
- Which customers are becoming inactive?
- Are there distinct customer behavior groups?
- How large and valuable is each segment?
- Can the segments support personalized retention and marketing actions?

### Important

RFM labels are a transparent business baseline.

K-Means is unsupervised and does not inherently know what "VIP", "at risk", or "loyal" means. Business labels are assigned only after inspecting cluster profiles.

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "customer_features.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "customer_segments.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature file: {DATA_PATH}")

## 1. Load and validate customer features

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run src/feature_engineering.py first."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

required_columns = [
    "customer_unique_id",
    "recency_days",
    "order_count",
    "total_revenue",
]

missing = [column for column in required_columns if column not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nRequired columns verified.")

## 2. Basic customer-feature diagnostics

In [ ]:
numeric_summary = df.select_dtypes(include=np.number).describe().T
display(numeric_summary)

missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

missing_summary["missing_pct"] = (
    100 * missing_summary["missing_values"] / len(df)
)

display(missing_summary.head(20))

## 3. RFM baseline

RFM means:

- **Recency** — how recently the customer purchased.
- **Frequency** — how often the customer purchased.
- **Monetary** — how much the customer spent.

The feature-engineering module already calculated RFM scores and a transparent business segment. We inspect that baseline before fitting an unsupervised model.

In [ ]:
rfm_columns = [
    "recency_score",
    "frequency_score",
    "monetary_score",
    "rfm_score",
    "customer_segment",
]

available_rfm = [column for column in rfm_columns if column in df.columns]

display(df[available_rfm].head())

if "customer_segment" in df.columns:
    rfm_distribution = (
        df["customer_segment"]
        .value_counts()
        .rename_axis("segment")
        .reset_index(name="customer_count")
    )

    rfm_distribution["percentage"] = (
        100 * rfm_distribution["customer_count"] / len(df)
    ).round(2)

    display(rfm_distribution)

    fig = px.bar(
        rfm_distribution,
        x="segment",
        y="customer_count",
        title="RFM Baseline Customer Segments",
        labels={"segment": "Segment", "customer_count": "Customers"},
    )
    fig.show()

## 4. RFM segment business profile

In [ ]:
if "customer_segment" in df.columns:
    rfm_profile = (
        df.groupby("customer_segment")
        .agg(
            customers=("customer_unique_id", "count"),
            average_recency_days=("recency_days", "mean"),
            average_orders=("order_count", "mean"),
            average_revenue=("total_revenue", "mean"),
            total_revenue=("total_revenue", "sum"),
            average_order_value=("average_order_value", "mean"),
            average_review_score=("average_review_score", "mean"),
        )
        .sort_values("total_revenue", ascending=False)
    )

    rfm_profile["revenue_share_pct"] = (
        100 * rfm_profile["total_revenue"] / rfm_profile["total_revenue"].sum()
    )

    display(rfm_profile.round(2))

## 5. Select features for K-Means

We avoid identifiers, raw dates, and already-created RFM/business labels.

The clustering model uses customer behavior such as:

- recency
- frequency
- monetary value
- product breadth
- review behavior
- delivery behavior

In [ ]:
candidate_features = [
    "recency_days",
    "order_count",
    "total_revenue",
    "total_items",
    "unique_products_purchased",
    "average_order_value",
    "average_review_score",
    "negative_review_rate",
    "average_delivery_days",
    "average_delivery_vs_estimate_days",
]

cluster_features = [
    column for column in candidate_features if column in df.columns
]

print("Clustering features:")
for column in cluster_features:
    print(f" - {column}")

X = df[cluster_features].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("\nFeature matrix:", X.shape)

## 6. Reduce skew before scaling

E-commerce monetary and frequency variables can be highly long-tailed.

For selected non-negative variables we use:

```text
log1p(x)
```

This reduces the dominance of extreme values while retaining those customers.

In [ ]:
X_model = X.copy()

log_features = [
    "recency_days",
    "order_count",
    "total_revenue",
    "total_items",
    "unique_products_purchased",
    "average_order_value",
    "average_delivery_days",
]

for column in log_features:
    if column in X_model.columns:
        X_model[column] = np.log1p(X_model[column].clip(lower=0))

display(X_model.describe().T)

## 7. Standardize the clustering matrix

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

print("Scaled matrix shape:", X_scaled.shape)
print("Mean approximately:", np.round(X_scaled.mean(axis=0), 4))
print("Std approximately:", np.round(X_scaled.std(axis=0), 4))

## 8. Select the number of clusters

We evaluate several values of `k` using silhouette score.

A higher silhouette score generally indicates better separation and cohesion, but business interpretability also matters.

In [ ]:
k_values = range(2, 9)
silhouette_results = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels,
        sample_size=min(10000, len(X_scaled)),
        random_state=42,
    )

    silhouette_results.append({
        "k": k,
        "silhouette_score": score,
    })

silhouette_df = pd.DataFrame(silhouette_results)

display(silhouette_df)

fig = px.line(
    silhouette_df,
    x="k",
    y="silhouette_score",
    markers=True,
    title="Silhouette Score by Number of Clusters",
    labels={"k": "Number of Clusters", "silhouette_score": "Silhouette Score"},
)
fig.show()

## 9. Select final K

In [ ]:
best_k = int(
    silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "k"]
)

best_score = float(
    silhouette_df.loc[
        silhouette_df["silhouette_score"].idxmax(),
        "silhouette_score",
    ]
)

print(f"Selected k: {best_k}")
print(f"Silhouette score: {best_score:.4f}")

## 10. Train final K-Means model

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10,
)

df["cluster_id"] = kmeans.fit_predict(X_scaled)

cluster_counts = (
    df["cluster_id"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster_id")
    .reset_index(name="customer_count")
)

display(cluster_counts)

## 11. Profile the clusters

In [ ]:
cluster_profile = (
    df.groupby("cluster_id")
    .agg(
        customers=("customer_unique_id", "count"),
        average_recency_days=("recency_days", "mean"),
        average_orders=("order_count", "mean"),
        average_revenue=("total_revenue", "mean"),
        total_revenue=("total_revenue", "sum"),
        average_order_value=("average_order_value", "mean"),
        average_items=("total_items", "mean"),
        average_unique_products=(
            "unique_products_purchased",
            "mean",
        ),
        average_review_score=("average_review_score", "mean"),
        average_delivery_days=("average_delivery_days", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

cluster_profile["customer_share_pct"] = (
    100 * cluster_profile["customers"] / len(df)
)

cluster_profile["revenue_share_pct"] = (
    100 * cluster_profile["total_revenue"] / cluster_profile["total_revenue"].sum()
)

display(cluster_profile.round(2))

## 12. Visualize cluster business value

In [ ]:
profile_for_plot = cluster_profile.reset_index()

fig = px.scatter(
    profile_for_plot,
    x="average_orders",
    y="average_revenue",
    size="customers",
    color="cluster_id",
    hover_data=[
        "average_recency_days",
        "average_order_value",
        "revenue_share_pct",
    ],
    title="Customer Clusters: Frequency vs Monetary Value",
    labels={
        "average_orders": "Average Orders",
        "average_revenue": "Average Revenue",
        "cluster_id": "Cluster",
    },
)
fig.show()

## 13. Cluster recency vs revenue

In [ ]:
fig = px.scatter(
    profile_for_plot,
    x="average_recency_days",
    y="average_revenue",
    size="customers",
    color="cluster_id",
    hover_data=[
        "average_orders",
        "average_order_value",
        "revenue_share_pct",
    ],
    title="Customer Clusters: Recency vs Revenue",
    labels={
        "average_recency_days": "Average Recency (days)",
        "average_revenue": "Average Revenue",
        "cluster_id": "Cluster",
    },
)
fig.show()

## 14. Assign business interpretations

Cluster IDs are arbitrary. Cluster `0` does not inherently mean "best".

We assign interpretations from the observed profiles:

- highest average revenue → **High Value**
- high value + oldest average recency → **At Risk High Value**
- most recent average recency → **Recently Active**
- remaining clusters → **Core / Developing**

These labels are a business interpretation layer, not part of K-Means.

In [ ]:
profile = cluster_profile.copy()

highest_value_cluster = profile["average_revenue"].idxmax()
most_recent_cluster = profile["average_recency_days"].idxmin()

high_value_candidates = profile[
    profile["average_revenue"] >= profile["average_revenue"].median()
]

if not high_value_candidates.empty:
    at_risk_cluster = high_value_candidates["average_recency_days"].idxmax()
else:
    at_risk_cluster = highest_value_cluster

business_labels = {}

for cluster_id in profile.index:
    if cluster_id == highest_value_cluster:
        label = "High Value"
    elif (
        cluster_id == at_risk_cluster
        and cluster_id != highest_value_cluster
    ):
        label = "At Risk High Value"
    elif cluster_id == most_recent_cluster:
        label = "Recently Active"
    else:
        label = "Core / Developing"

    business_labels[cluster_id] = label

df["segment"] = df["cluster_id"].map(business_labels)

segment_summary = (
    df.groupby(["cluster_id", "segment"])
    .agg(
        customers=("customer_unique_id", "count"),
        average_recency_days=("recency_days", "mean"),
        average_orders=("order_count", "mean"),
        average_revenue=("total_revenue", "mean"),
        total_revenue=("total_revenue", "sum"),
        average_order_value=("average_order_value", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

segment_summary["customer_share_pct"] = (
    100 * segment_summary["customers"] / len(df)
)

segment_summary["revenue_share_pct"] = (
    100 * segment_summary["total_revenue"]
    / segment_summary["total_revenue"].sum()
)

display(segment_summary.round(2))

## 15. Compare RFM and K-Means segments

In [ ]:
if "customer_segment" in df.columns:
    comparison = pd.crosstab(
        df["customer_segment"],
        df["segment"],
        normalize="index",
    ) * 100

    display(comparison.round(2))

## 16. Save the segmentation dataset and model

The dataset is saved for downstream recommendation, retention, dashboard, and AI-insight components.

The model artifact stores the K-Means model and the exact preprocessing required during inference.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_PATH,
    index=False,
)

MODEL_PATH = MODEL_DIR / "customer_segmentation.joblib"

model_artifact = {
    "model": kmeans,
    "scaler": scaler,
    "features": cluster_features,
    "log_features": [
        column for column in log_features
        if column in cluster_features
    ],
    "business_labels": business_labels,
    "selected_k": best_k,
    "silhouette_score": best_score,
}

joblib.dump(model_artifact, MODEL_PATH)

print(f"Saved segmented data: {OUTPUT_PATH}")
print(f"Saved model artifact: {MODEL_PATH}")

# Final validation

Verify that these two artifacts exist:

```text
data/processed/customer_segments.csv
models/customer_segmentation.joblib
```

The saved model contains:

- K-Means model
- StandardScaler
- feature list
- log-transformed feature list
- selected number of clusters
- silhouette score
- business-label mapping

This makes later inference reproducible.